# CADForge OpenEnv Training Notebook

This notebook is the judge-runnable entry point for CADForge. It wraps the same production scripts used for the RunPod H200 training run:

- OpenEnv validation for `experiment-2-cadforge`
- Unsloth LoRA SFT smoke or full training
- TRL GRPO against the CADForge reward environment
- evaluation through real CadQuery build/scoring
- plot/report generation

The full run can take hours on an H200. The smoke path is intentionally small so judges can verify that the pipeline connects to the environment without spending full training time.

## 1. Configure

Set `RUN_FULL_TRAINING = True` only on a real GPU runtime. The smoke path runs a few steps and proves that the model, data, CADForge reward, and reports all connect.

In [ ]:
import os

REPO_URL = "https://github.com/sanjuhs/open-env-meta-final-hackathon.git"
REPO_DIR = "/content/open-env-meta-final-hackathon"

# Optional. Required for pushing model adapters or downloading private assets.
os.environ.setdefault("HF_TOKEN", "")

# Keep full training off by default for judge smoke reruns.
RUN_FULL_TRAINING = False

# Model choices used in the final run.
BASE_2B = "unsloth/Qwen3.5-2B"
BASE_9B = "unsloth/Qwen3.5-9B"

# Public dataset produced for CADForge.
DATASET = "sanjuhs/cadforge-cadquery-agentic-traces"


## 2. Clone and Install

This uses `uv` so the notebook matches the production RunPod scripts. On a local checkout, change `REPO_DIR` to your repo path and skip the clone.

In [ ]:
%%bash
set -euo pipefail

if [ ! -d /content/open-env-meta-final-hackathon ]; then
  git clone https://github.com/sanjuhs/open-env-meta-final-hackathon.git /content/open-env-meta-final-hackathon
fi

cd /content/open-env-meta-final-hackathon
curl -LsSf https://astral.sh/uv/install.sh | sh || true
export PATH="$HOME/.local/bin:$PATH"

export UV_CACHE_DIR=/content/.uv-cache
export HF_HOME=/content/.cache/huggingface
export TORCH_HOME=/content/.cache/torch
export TRITON_CACHE_DIR=/content/.cache/triton
export VLLM_CACHE_ROOT=/content/.cache/vllm
export UV_LINK_MODE=copy
export HF_HUB_ENABLE_HF_TRANSFER=1
export HF_HUB_DISABLE_XET=1

uv sync --project experiment-2-cadforge


## 3. Validate OpenEnv and Smoke the Reward

This confirms that the submission is an OpenEnv app and that the CadQuery reward backend can compile and score a known candidate.

In [ ]:
%%bash
set -euo pipefail
cd /content/open-env-meta-final-hackathon
export PATH="$HOME/.local/bin:$PATH"

uv run --with openenv-core[core] openenv validate experiment-2-cadforge
uv run training/smoke_cadforge_reward.py --reward-mode fast


## 4. SFT Smoke Test

SFT teaches Qwen the language of editable CadQuery: named dimensions, helper functions, Python-only output, and final `fixture` assignment.

In [ ]:
%%bash
set -euo pipefail
cd /content/open-env-meta-final-hackathon
export PATH="$HOME/.local/bin:$PATH"
export HF_HUB_DISABLE_XET=1

uv run training/train_sft_unsloth.py \
  --model unsloth/Qwen3.5-2B \
  --output-dir outputs/qwen35-2b-cadforge-sft-smoke \
  --max-steps 10 \
  --limit-train-rows 32 \
  --limit-val-rows 8 \
  --max-seq-length 4096 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 4 \
  --lora-r 16 \
  --lora-alpha 32 \
  --run-name qwen35-2b-sft-smoke


## 5. GRPO Smoke Test Against CADForge

This is the RLVE part: completions are scored by the real CadQuery environment. The strict gate makes failed builds negative and successful builds unlock dense CAD rewards.

In [ ]:
%%bash
set -euo pipefail
cd /content/open-env-meta-final-hackathon
export PATH="$HOME/.local/bin:$PATH"
export HF_HUB_DISABLE_XET=1
export CADFORGE_PYTHON=/content/open-env-meta-final-hackathon/experiment-2-cadforge/.venv/bin/python

uv run training/train_grpo_cadforge.py \
  --model unsloth/Qwen3.5-2B \
  --adapter outputs/qwen35-2b-cadforge-sft-smoke \
  --output-dir outputs/qwen35-2b-cadforge-grpo-smoke \
  --reward-backend cadforge \
  --strict-build-gate \
  --cadforge-python "$CADFORGE_PYTHON" \
  --cadforge-reward-mode fast \
  --limit-prompts 4 \
  --max-steps 3 \
  --num-generations 2 \
  --max-prompt-length 4096 \
  --max-completion-length 1536 \
  --max-seq-length 6144 \
  --per-device-train-batch-size 2 \
  --gradient-accumulation-steps 1 \
  --learning-rate 5e-6 \
  --debug-completions-jsonl training/logs/notebook-grpo-smoke-completions.jsonl \
  --run-name notebook-qwen35-2b-grpo-smoke


## 6. Full Training Commands Used for the Submission

The final strict run used the 9B SFT checkpoint, `80` GRPO optimizer steps, and strict build gating. Keep this cell disabled unless you are on an H200/H100-class GPU.

In [ ]:
if RUN_FULL_TRAINING:
    raise SystemExit("Run the production shell pipeline on a dedicated H200/RunPod session: training/cadforge_training_pipeline.sh and training/run_strict_9b_grpo.sh")
else:
    print("Full training disabled. See training/cadforge_training_pipeline.sh and docs/brainstorm/20-cadforge-qwen-training-runbook.md.")


## 7. Generate Charts and View Evidence

The production run already produced these committed reports. If you rerun training, regenerate them with `training/make_training_report.py`.

In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path

repo = Path('/content/open-env-meta-final-hackathon')
report_dir = repo / 'training/reports/qwen35-9b-grpo-strict-build-20260426-strict-build'

for image_name in ['grpo_reward_curve.png', 'grpo_code_health.png', 'grpo_error_breakdown.png']:
    path = report_dir / image_name
    if path.exists():
        display(Markdown(f'### {image_name}'))
        display(Image(filename=str(path)))
    else:
        print('Missing', path)


## 8. Final Run Metrics

These are the metrics from the actual H200 run submitted with CADForge.

In [ ]:
import json
from pathlib import Path

repo = Path('/content/open-env-meta-final-hackathon')
eval_report = repo / 'training/eval/qwen35-9b-cadforge-grpo-strict-build-20260426-strict-build/eval_report.md'
training_report = repo / 'training/reports/qwen35-9b-grpo-strict-build-20260426-strict-build/training_curve_report.md'

if training_report.exists():
    display(Markdown(training_report.read_text()))
if eval_report.exists():
    display(Markdown(eval_report.read_text()))
